In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from scipy import stats
import joblib
import os
import time
from datetime import datetime

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LinearRegression,  Ridge, Lasso
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, RandomForestClassifier
from sklearn.feature_selection import RFE, RFECV

In [2]:
#pip install openpyxl

In [3]:
customer_raw = pd.read_csv("musteri-satis-datasi.csv", encoding="latin-1", sep=";")
order_raw = pd.read_excel("Order_table.xlsx")
product_raw = pd.read_excel("products.xlsx")

print(f"Customer Dataset loaded: {customer_raw.shape[0]} rows and {customer_raw.shape[1]} columns")
print(f"Order Dataset loaded: {order_raw.shape[0]} rows and {order_raw.shape[1]} columns")
print(f"Product Dataset loaded: {order_raw.shape[0]} rows and {order_raw.shape[1]} columns")

Customer Dataset loaded: 18070 rows and 6 columns
Order Dataset loaded: 102885 rows and 8 columns
Product Dataset loaded: 102885 rows and 8 columns


## TABLE 1: Customers

In [4]:
customer_raw.head(5)

,Üye Kodu,Üye Adý,Sipariþ Sayýsý,Ürün Adet,Tutar,Üye Kayýt Tarihi
0,T10002,F*** Y***,2,3,"2731,40",1.01.1970
1,T10008,A*** Y***,4,134,"8537,99",1.01.1970
2,T1002,A*** U***,2,3,"4234,00",1.01.1970
3,T10060,Þ*** A***,4,4,"10434,45",1.01.1970
4,T10061,S*** G***,34,494,"56770,06",1.01.1970


In [5]:
customer_raw = customer_raw.drop(columns="Üye Adý")

In [6]:
# Step 1: Rename columns to be shorter and cleaner
customers = customer_raw.rename(columns={
    'Üye Kodu': 'customer_id',
    'Sipariþ Sayýsý': 'total_orders',
    'Ürün Adet': 'total_products_bought',
    'Tutar': 'total_spend_tl',
    'Üye Kayýt Tarihi': 'registration_date'
})

print("Renamed columns:")
print(customers.columns.tolist())

Renamed columns:
['customer_id', 'total_orders', 'total_products_bought', 'total_spend_tl', 'registration_date']


In [7]:
# Step 2: Check data types and nulls
print("Data types:")
print(customers.dtypes)
print("\nNull counts:")
print(customers.isnull().sum())

Data types:
customer_id              object
total_orders              int64
total_products_bought     int64
total_spend_tl           object
registration_date        object
dtype: object

Null counts:
customer_id              0
total_orders             0
total_products_bought    0
total_spend_tl           0
registration_date        0
dtype: int64


In [8]:
customers.head()

,customer_id,total_orders,total_products_bought,total_spend_tl,registration_date
0,T10002,2,3,"2731,40",1.01.1970
1,T10008,4,134,"8537,99",1.01.1970
2,T1002,2,3,"4234,00",1.01.1970
3,T10060,4,4,"10434,45",1.01.1970
4,T10061,34,494,"56770,06",1.01.1970


In [9]:
# Step 3: Fix total_spend_tl & registration_date
# Turkish number format uses '.' as thousands separator and ',' as decimal
customers['total_spend_tl'] = (
    customers['total_spend_tl']
    .str.replace('.', '', regex=False)   # remove thousands separator
    .str.replace(',', '.', regex=False)  # make decimal point standard
    .astype(float)
)

# format='%d.%m.%Y' means: day.month.4-digit-year → matches '16.04.2024'
customers['registration_date'] = pd.to_datetime(
    customers['registration_date'],
    format='%d.%m.%Y'
)

# Verify
print(customers[['total_spend_tl', 'registration_date']].dtypes)
print()
print(customers[['total_spend_tl', 'registration_date']].head())

total_spend_tl              float64
registration_date    datetime64[ns]
dtype: object

   total_spend_tl registration_date
0         2731.40        1970-01-01
1         8537.99        1970-01-01
2         4234.00        1970-01-01
3        10434.45        1970-01-01
4        56770.06        1970-01-01


In [10]:
# Step 4: Identify pre-migration customers 
pre_migration_mask = customers['registration_date'].dt.year == 1970
print(f'Pre-migration customers (1970 date): {pre_migration_mask.sum()}')
print(f'Customers with valid registration date: {(~pre_migration_mask).sum()}')

Pre-migration customers (1970 date): 835
Customers with valid registration date: 17235


In [11]:
# Step 5: Quick checks
print("Basic statistics for numeric columns:")
print(customers[['total_orders', 'total_products_bought', 'total_spend_tl']].describe())

Basic statistics for numeric columns:
       total_orders  total_products_bought  total_spend_tl
count  18070.000000           18070.000000    1.807000e+04
mean       2.434975               9.878251    7.963522e+03
std       15.710486             100.029960    7.315999e+04
min        1.000000               1.000000    3.559000e+01
25%        1.000000               1.000000    1.383685e+03
50%        1.000000               2.000000    3.724000e+03
75%        2.000000               6.000000    6.964788e+03
max     1340.000000            7407.000000    8.490122e+06


In [12]:
# Step 6: Create a new column identifying customer tenure in years
SNAPSHOT_DATE = pd.Timestamp('2026-02-09')
# Set 1970 dates to NaT (missing) before calculating tenure
customers.loc[pre_migration_mask, 'registration_date'] = pd.NaT

# Calculate tenure in years from SNAPSHOT_DATE
customers['tenure_years'] = (
    (SNAPSHOT_DATE - customers['registration_date']).dt.days / 365.25
).round(2)

# Assign tenure bucket
# - 'pre_system': registered before the system migration (date unknown)
#                 These are likely your most loyal, longest-standing customers.
# - 'less_than_1yr', 'one_to_2yr', 'two_to_3yr', 'over_3yr': for known dates
def assign_tenure_bucket(row):
    if pd.isna(row['registration_date']):
        return '>3yr'
    elif row['tenure_years'] < 1:
        return '<1'
    elif row['tenure_years'] < 2:
        return '1_to_2yr'
    elif row['tenure_years'] < 3:
        return '2_to_3yr'
    else:
        return '>3yr'

customers['tenure_bucket'] = customers.apply(assign_tenure_bucket, axis=1)

print('Tenure bucket distribution:')
print(customers['tenure_bucket'].value_counts())

Tenure bucket distribution:
tenure_bucket
<1          7876
1_to_2yr    6486
2_to_3yr    2791
>3yr         917
Name: count, dtype: int64


In [13]:
# Step 7: Check for any customers where total_products < total_orders
# This happens when customers returned items, a return reduces the product count
# but the original order still exists in the system.
# We keep these rows, they're valid customers. We just note the pattern.
returns_likely = customers[customers['total_products_bought'] < customers['total_orders']]
print(f'Customers where products < orders (likely had returns): {len(returns_likely)}')
print(returns_likely[['customer_id', 'total_orders', 'total_products_bought', 'total_spend_tl']])
print()
print('NOTE: We keep these customers. The products < orders difference')
print('tells us how many net returns they made — potentially useful as a feature.')

Customers where products < orders (likely had returns): 315
      customer_id  total_orders  total_products_bought  total_spend_tl
29          T1034             4                      2         3843.00
51         T10674            12                     11        36136.95
89         T11182             3                      1         2356.00
101        T11273             3                      2        11660.00
139        T11642            67                     47       121577.68
...           ...           ...                    ...             ...
17905       T7907             2                      1         2080.00
17906       T7908            26                     25        92632.00
17960       T8684             5                      3        11802.00
17974       T8803             8                      7        11220.49
18051       T9775             7                      6         7837.00

[315 rows x 4 columns]

NOTE: We keep these customers. The products < orders difference

In [14]:
# Step 8: Create a return indicator columns
# est_returns: how many net returned line items (orders - products bought, floored at 0)
# has_returned: simple True/False flag — easier to use in models

customers['est_returns'] = (
    customers['total_orders'] - customers['total_products_bought']
).clip(lower=0)

customers['has_returned'] = customers['est_returns'] > 0

# Sanity check
print(f"Customers with at least 1 return: {customers['has_returned'].sum()}")
print(f"That's {customers['has_returned'].mean()*100:.1f}% of all customers")
print()
print(customers[customers['has_returned']][['customer_id', 'total_orders', 'total_products_bought', 'est_returns']].head())

Customers with at least 1 return: 315
That's 1.7% of all customers

    customer_id  total_orders  total_products_bought  est_returns
29        T1034             4                      2            2
51       T10674            12                     11            1
89       T11182             3                      1            2
101      T11273             3                      2            1
139      T11642            67                     47           20


In [15]:
customers = customers.drop(columns=["registration_date","tenure_years"])

In [16]:
customers.head()

,customer_id,total_orders,total_products_bought,total_spend_tl,tenure_bucket,est_returns,has_returned
0,T10002,2,3,2731.40,>3yr,0,False
1,T10008,4,134,8537.99,>3yr,0,False
2,T1002,2,3,4234.00,>3yr,0,False
3,T10060,4,4,10434.45,>3yr,0,False
4,T10061,34,494,56770.06,>3yr,0,False


## TABLE 2: Orders

In [17]:
order_raw.head()

,T-Soft Üye Kodu,ID,Sipariş No,T-Soft Ürün Kodu,Ürün Adı,Adet,Ürün Birim Fiyatı,Tarih
0,T24540,35579,TS010135579,362,Clinique For Men Kırışık Karşıtı Göz Kremi 15 Ml,1,756.0,2024-01-01
1,T24667,35506,TS010135506,1212,Lancome Hypnose Kadın Parfüm Edp 75 Ml,1,2990.0,2024-01-01
2,T24668,35510,TS010135510,1970,Carolina Herrera 212 Sexy Men Erkek Parfüm Edt...,1,2449.6,2024-01-01
3,ts_1704131707_8429,35585,TS010135585,1970,Carolina Herrera 212 Sexy Men Erkek Parfüm Edt...,1,2449.6,2024-01-01
4,T1186,35548,TS010135548,3480,YSL La Nuit De L'Homme Erkek Deo Stick 75 Gr,1,916.5,2024-01-01


In [18]:
order_raw = order_raw.drop(columns="ID")

In [19]:
# Step 1: Rename columns
orders = order_raw.rename(columns={
    'T-Soft Üye Kodu': 'customer_id',
    'Sipariş No': 'order_id',
    'T-Soft Ürün Kodu': 'product_code',
    'Ürün Adı': 'product_name',
    'Adet': 'quantity',
    'Ürün Birim Fiyatı': 'unit_price',
    'Tarih': 'purchase_date'
})

print("Renamed columns:")
print(customers.columns.tolist())

Renamed columns:
['customer_id', 'total_orders', 'total_products_bought', 'total_spend_tl', 'tenure_bucket', 'est_returns', 'has_returned']


In [20]:
# Step 2: Check data types
print("Data types:")
print(orders.dtypes)
print("\nNull counts:")
print(orders.isnull().sum())

Data types:
customer_id              object
order_id                 object
product_code             object
product_name             object
quantity                  int64
unit_price              float64
purchase_date    datetime64[ns]
dtype: object

Null counts:
customer_id         0
order_id            0
product_code     5022
product_name        0
quantity            0
unit_price          0
purchase_date       0
dtype: int64


In [21]:
# Step 3: Check for unknown/error product codes
unknown_codes = orders[orders['product_code'] == 'UNKNOWN_DATE_PARSE_ERROR']
print(f"Rows with unrecoverable product codes: {len(unknown_codes)}")
print(unknown_codes[['order_id', 'product_code', 'product_name', 'quantity', 'unit_price']])

# DECISION: We keep these rows. The product name is intact, so we can
# potentially match them to the product catalog by name later in Phase 2.
# We do NOT drop them — dropping transaction data loses revenue signal.

Rows with unrecoverable product codes: 0
Empty DataFrame
Columns: [order_id, product_code, product_name, quantity, unit_price]
Index: []


In [22]:
# Step 4: Remove RETURN rows (negative quantities)
# Negative quantity means a product was returned.
# We remove these from the orders table because:
# (a) Returns are already baked into the customer summary table (total_products_bought)
# (b) Keeping them would create phantom negative transactions in our interaction matrix
return_mask = orders['quantity'] <= 0
print(f'Return rows (quantity <= 0) to remove: {return_mask.sum()}')
if return_mask.sum() > 0:
    print(orders[return_mask][['order_id', 'product_code', 'product_name', 'quantity']].head(10))

orders = orders[~return_mask].copy()
print(f'Rows after removing returns: {len(orders)}')

Return rows (quantity <= 0) to remove: 0
Rows after removing returns: 102885


In [23]:
# Step 5: Fix price floating-point noise
# Rounding to 2 decimal places (cents) fixes this cleanly.
orders['unit_price'] = orders['unit_price'].round(2)

# Add a line_total column — the total cost for that line item
orders['line_total'] = (orders['quantity'] * orders['unit_price']).round(2)

print("Price sample after rounding:")
print(orders[['product_name', 'quantity', 'unit_price', 'line_total']].head(10))

Price sample after rounding:
                                        product_name  quantity  unit_price  \
0   Clinique For Men Kırışık Karşıtı Göz Kremi 15 Ml         1      756.00   
1             Lancome Hypnose Kadın Parfüm Edp 75 Ml         1     2990.00   
2  Carolina Herrera 212 Sexy Men Erkek Parfüm Edt...         1     2449.60   
3  Carolina Herrera 212 Sexy Men Erkek Parfüm Edt...         1     2449.60   
4       YSL La Nuit De L'Homme Erkek Deo Stick 75 Gr         1      916.50   
5   Giorgio Armani Code Femme Kadın Parfüm Edp 75 Ml         1     2996.50   
6  Giorgio Armani Acqua Di Gioia Kadın Parfüm Edp...         1     2957.50   
7                       Taft Ultra Saç Spreyi 250 Ml         1       93.67   
8         Vi-Vet Yüz Sir Ağda Bandı Portakallı 24'lü         1       41.50   
9                         Solingen Tırnak Makası 312         1       60.30   

   line_total  
0      756.00  
1     2990.00  
2     2449.60  
3     2449.60  
4      916.50  
5     2996.50  


In [24]:
# Step 7: Strip time from purchase_date — we only need the date
orders['purchase_date'] = pd.to_datetime(orders['purchase_date']).dt.normalize()

# Step 8: Check for any negative quantities or prices with negative values of 0 values (data entry errors)
print(f"Rows with quantity <= 0: {(orders['quantity'] <= 0).sum()}")
print(f"Rows with unit_price <= 0: {(orders['unit_price'] <= 0).sum()}")

Rows with quantity <= 0: 0
Rows with unit_price <= 0: 124


In [25]:
# Create column to indicate whether it is a gift (free item logic)
# Gift = item with zero (or negative) price
#orders['is_gift'] = orders['unit_price'] <= 0
# OR gift at order level: if any item in the order is free
orders['is_gift'] = orders.groupby('order_id')['unit_price'].transform(lambda x: (x <= 0).any())

In [26]:
# Step 9: Check how many unique orders and customers we have
print(f"Unique order IDs: {orders['order_id'].nunique()}")
print(f"Unique customer IDs in orders: {orders['customer_id'].nunique()}")
print(f"Unique product codes: {orders['product_code'].nunique()}")
print(f"\nOrders table shape: {orders.shape}")

print("\nItems per order distribution:")
items_per_order = orders.groupby('order_id')['product_code'].count()
print(items_per_order.describe())

Unique order IDs: 53435
Unique customer IDs in orders: 34199
Unique product codes: 10171

Orders table shape: (102885, 9)

Items per order distribution:
count    53435.000000
mean         1.831440
std          2.009879
min          0.000000
25%          1.000000
50%          1.000000
75%          2.000000
max         72.000000
Name: product_code, dtype: float64


In [27]:
# Step 10: Identify unpurchased products
# We have 15k products but only ~11k appear in orders.
# The ~4k that never appear are 'cold start' products — 
# collaborative filtering can't recommend them because there's no purchase history.
# We'll handle these with content-based fallback in Phase 2 (using category/brand similarity).
purchased_codes = set(orders['product_code'].unique())
print(f'Unique product codes in orders: {len(purchased_codes)}')
# We'll compare against the products table after we clean it below.

Unique product codes in orders: 10172


In [28]:
# Step 11: Flag sub-products (dashed codes like '400-425')
# A dashed code means this is a variant (e.g., mascara in color #425 from parent SKU 400).
# We KEEP these, they're real purchases. But flagging them is useful:
# - For collaborative filtering: sub-products of the same parent are very similar items
# - For market basket analysis: buying variant A doesn't strongly predict buying variant B
#   of the same product (it's the same product in a different color)
orders['is_subproduct'] = orders['product_code'].str.match(r'^\d+-\d+$', na=False)

print(f"Sub-product rows: {orders['is_subproduct'].sum()}")
print(f"Standard product rows: {(~orders['is_subproduct']).sum()}")

Sub-product rows: 25603
Standard product rows: 77282


create is_member column to identify the members from the ones who purchased products withot becoming a member

In [29]:
# Step 12: Create is_member column
# Members: customer_id starts with 'T'
# Non-members: customer_id starts with 'ts_'

orders['is_member'] = orders['customer_id'].astype(str).str.startswith('T')

In [30]:
# Step 13: Clean rows with product code being NA
orders = orders.dropna(subset=['product_code'])

In [31]:
# Check missing product codes
print("Missing product_code count:", orders['product_code'].isna().sum())

# Percentage missing
print("Missing %:", orders['product_code'].isna().mean())

# See examples
orders[orders['product_code'].isna()].head()

Missing product_code count: 0
Missing %: 0.0


,customer_id,order_id,product_code,product_name,quantity,unit_price,purchase_date,line_total,is_gift,is_subproduct,is_member


In [32]:
# Final clean orders table
print("CLEAN ORDERS TABLE:")
print(f"Shape: {orders.shape}")
orders.head(15)

CLEAN ORDERS TABLE:
Shape: (97863, 11)


,customer_id,order_id,product_code,product_name,quantity,unit_price,purchase_date,line_total,is_gift,is_subproduct,is_member
0,T24540,TS010135579,362,Clinique For Men Kırışık Karşıtı Göz Kremi 15 Ml,1,756.00,2024-01-01,756.00,False,False,True
1,T24667,TS010135506,1212,Lancome Hypnose Kadın Parfüm Edp 75 Ml,1,2990.00,2024-01-01,2990.00,False,False,True
2,T24668,TS010135510,1970,Carolina Herrera 212 Sexy Men Erkek Parfüm Edt...,1,2449.60,2024-01-01,2449.60,False,False,True
3,ts_1704131707_8429,TS010135585,1970,Carolina Herrera 212 Sexy Men Erkek Parfüm Edt...,1,2449.60,2024-01-01,2449.60,False,False,False
4,T1186,TS010135548,3480,YSL La Nuit De L'Homme Erkek Deo Stick 75 Gr,1,916.50,2024-01-01,916.50,False,False,True
5,T8144,TS010135552,4633,Giorgio Armani Code Femme Kadın Parfüm Edp 75 Ml,1,2996.50,2024-01-01,2996.50,False,False,True
6,T24674,TS010135527,4713,Giorgio Armani Acqua Di Gioia Kadın Parfüm Edp...,1,2957.50,2024-01-01,2957.50,False,False,True
7,T24678,TS010135540,5939,Taft Ultra Saç Spreyi 250 Ml,1,93.67,2024-01-01,93.67,False,False,True
8,T24680,TS010135558,11193,Vi-Vet Yüz Sir Ağda Bandı Portakallı 24'lü,1,41.50,2024-01-01,41.50,False,False,True
9,T24683,TS010135563,13008,Solingen Tırnak Makası 312,1,60.30,2024-01-01,60.30,False,False,True


## TABLE 3: Products

In [48]:
product_raw.head()

,Product Name,Web Service Code,General Category,Middle Category,Sub Category,Category Full Path,Selling Price,Discounted Price,Brand
0,Sesu Sir Ağda Bantları Normal Tüyler 12'li,24568,Kişisel Bakım,Epilasyon Ürünleri,Ağda Bantları,Kişisel Bakım > Epilasyon Ürünleri > Ağda Bant...,0.0,0.0,Sesu
1,Sesu Vücut Ağda Bandı Kalın Tüy 12'li,24569,Kişisel Bakım,Epilasyon Ürünleri,Ağda Bantları,Kişisel Bakım > Epilasyon Ürünleri > Ağda Bant...,0.0,0.0,Sesu
2,YSL La Vestiaire Universite Unisex Parfüm Edp ...,117829,Parfüm,Unisex Parfüm,EDP Parfüm,Parfüm > Unisex Parfüm >EDP Parfüm,0.0,0.0,YSL
3,Pastel Show By Eyeliner 124,1458-907,Makyaj,Göz,Eyeliner,Makyaj > Göz > Eyeliner,0.0,0.0,Pastel
4,L'Oréal Paris Infaillible 24H Matte Cover Foun...,1135-1044,Makyaj,Yüz,Fondöten,Makyaj > Yüz > Fondöten,0.0,0.0,Loreal Paris Makyaj


In [49]:
# Step 1: Rename columns
products = product_raw.rename(columns={
    'Product Name': 'product_name',
    'Web Service Code': 'product_code',   # matches the 'product_code' in orders
    'General Category': 'category_main',
    'Middle Category': 'category_mid',
    'Sub Category': 'category_sub',
    'Category Full Path': 'full_path',
    'Selling Price': 'price_original',
    'Discounted Price': 'price_discounted',
    'Brand': 'brand'
})

# Note: we renamed 'Web Service Code' to 'product_code' because this is the code that appears in the Orders table
print("Columns renamed.")
print(products.columns.tolist())

Columns renamed.
['product_name', 'product_code', 'category_main', 'category_mid', 'category_sub', 'full_path', 'price_original', 'price_discounted', 'brand']


In [50]:
products.head()

,product_name,product_code,category_main,category_mid,category_sub,full_path,price_original,price_discounted,brand
0,Sesu Sir Ağda Bantları Normal Tüyler 12'li,24568,Kişisel Bakım,Epilasyon Ürünleri,Ağda Bantları,Kişisel Bakım > Epilasyon Ürünleri > Ağda Bant...,0.0,0.0,Sesu
1,Sesu Vücut Ağda Bandı Kalın Tüy 12'li,24569,Kişisel Bakım,Epilasyon Ürünleri,Ağda Bantları,Kişisel Bakım > Epilasyon Ürünleri > Ağda Bant...,0.0,0.0,Sesu
2,YSL La Vestiaire Universite Unisex Parfüm Edp ...,117829,Parfüm,Unisex Parfüm,EDP Parfüm,Parfüm > Unisex Parfüm >EDP Parfüm,0.0,0.0,YSL
3,Pastel Show By Eyeliner 124,1458-907,Makyaj,Göz,Eyeliner,Makyaj > Göz > Eyeliner,0.0,0.0,Pastel
4,L'Oréal Paris Infaillible 24H Matte Cover Foun...,1135-1044,Makyaj,Yüz,Fondöten,Makyaj > Yüz > Fondöten,0.0,0.0,Loreal Paris Makyaj


In [51]:
products['category_main'].value_counts(dropna=False).head(10)

category_main
Makyaj            5793
Parfüm            2658
Saç Bakım         1826
Kişisel Bakım     1703
Cilt Bakım        1131
Tekstil           1116
NaN                229
Erkek Bakım        178
Bebek              150
Name: count, dtype: int64

In [52]:
# Step 2: Check data types and nulls
print("Data types:")
print(products.dtypes)
print("\nNull counts:")
print(products.isnull().sum())

Data types:
product_name         object
product_code         object
category_main        object
category_mid         object
category_sub         object
full_path            object
price_original      float64
price_discounted    float64
brand                object
dtype: object

Null counts:
product_name          0
product_code          0
category_main       229
category_mid        728
category_sub          0
full_path             0
price_original        0
price_discounted      0
brand                13
dtype: int64


In [53]:
# Step 3: Standardize product_code to string
products['product_code'] = products['product_code'].astype(str).str.strip()

In [54]:
# Step 4: Clean whitespace from ALL string columns at once
# The category columns have leading/trailing spaces ('Parfüm ' vs 'Parfüm')
# This would cause groupby operations to treat them as different categories!

string_cols = ['product_name', 'category_main', 'category_mid', 'category_sub', 'brand']

for col in string_cols:
    products[col] = products[col].astype(str).str.strip()

print("Unique values in category_main after stripping:")
print(sorted(products['category_main'].unique()))

Unique values in category_main after stripping:
['Bebek', 'Cilt Bakım', 'Erkek Bakım', 'Kişisel Bakım', 'Makyaj', 'Parfüm', 'Saç Bakım', 'Tekstil', 'nan']


In [55]:
# Step 5: Fix the '\xa0' (non-breaking space) in category_mid
# \xa0 is an invisible HTML-style space character. str.strip() doesn't remove it.
# We need to explicitly replace it.

print("Before fix — unique category_mid values:")
print(products['category_mid'].unique())

# Replace non-breaking space with a regular space, then strip again
products['category_mid'] = products['category_mid'].str.replace('\xa0', ' ', regex=False).str.strip()

print("\nAfter fix — unique category_mid values:")
print(sorted(products['category_mid'].unique()))

Before fix — unique category_mid values:
['Epilasyon Ürünleri' 'Unisex Parfüm' 'Göz' 'Yüz' 'nan' '' 'Dudak'
 'Saç Bakım Ürünleri' 'Banyo Bakım' 'Ağız Bakım' 'Bebek Bakım Ürünleri'
 'Temizleyiciler' 'Nemlendirici' 'Vücut Bakım' 'Kadın Giyim'
 'Kadın İç Giyim' 'Saç Boyaları' 'Tırnak' 'Saç Şekillendirici'
 'Kadın Bakım' 'Saç Aksesuarları' 'Erkek Plaj Giyim' 'Kadın Parfüm'
 'Erkek Giyim' 'Tıraş Bıçakları' 'Tıraş Sonrası Ürünler'
 'Tıraş Öncesi Ürünler' 'Erkek Parfüm' 'Cilt Bakım' 'Kadın Plaj Giyim'
 'Yaşlanma Karşıtı' 'Çocuk Giyim']

After fix — unique category_mid values:
['', 'Ağız Bakım', 'Banyo Bakım', 'Bebek Bakım Ürünleri', 'Cilt Bakım', 'Dudak', 'Epilasyon Ürünleri', 'Erkek Giyim', 'Erkek Parfüm', 'Erkek Plaj Giyim', 'Göz', 'Kadın Bakım', 'Kadın Giyim', 'Kadın Parfüm', 'Kadın Plaj Giyim', 'Kadın İç Giyim', 'Nemlendirici', 'Saç Aksesuarları', 'Saç Bakım Ürünleri', 'Saç Boyaları', 'Saç Şekillendirici', 'Temizleyiciler', 'Tıraş Bıçakları', 'Tıraş Sonrası Ürünler', 'Tıraş Öncesi Ürünler

In [56]:
# Step 5: Investigate nan in category_main
nan_cat = products[products['category_main'].isna() | (products['category_main'].astype(str).str.strip() == 'nan')]
print(f'Products with nan category_main: {len(nan_cat)}')
if len(nan_cat) > 0:
    print(nan_cat[['product_code', 'product_name', 'category_main', 'category_sub']].head(20))
    print()
    # Check if these overlap with T-codes (already removed) or are something else
    print('Product codes for nan-category products:')
    print(nan_cat['product_code'].head(20).tolist())

Products with nan category_main: 229
     product_code                                       product_name  \
16         130231                             La Mer Renewal Oil Set   
17         132424  Dior Jadore Kadın Parfüm Edp 100 Ml+Edp 10 Ml Set   
3006       122241  Colgate Barbie-Batman Diş Macunu 75 Ml + Fırça...   
3761       123368  Colgate Üçlü Etki Diş Macunu 125 Ml + 125 Ml 2...   
3803       124621  Nivea Yoğun Nemlendirici El Bakım Kremi 75 Ml ...   
4192       120943    Dove Original Deo Spray + Kotex Normal 8'li Set   
5157       109022  Nivea Black&White Invisible Deo 150 Ml+Mini Ro...   
5158       127955  Nivea Fresh Natural Kadın Deodorant 150 Ml + C...   
5159       127956  Nivea Fresh Erkek Sprey Deodorant 150 Ml + Spo...   
5547       125694  Rebul Charlotte Kadın Parfüm Edp 100 Ml+Edp 20 Ml   
5548       125695    Rebul Scarlet Kadın Parfüm Edp 100 Ml+Edp 20 Ml   
5549       125696   Rebul Isabella Kadın Parfüm Edp 100 Ml+Edp 20 Ml   
5550       125697     Rebul

For encoding the category path -> into 3 different columsn to preserve the hierarchical taxonomy:
- If only sub exists → it is the true category
- If main exists but mid missing → mid = “NA”
- If only sub filled → propagate hierarchy if needed

Therefore we have this structure:
- Some products have 3 levels (main → mid → sub)
- Some have 2 levels
- Some have only 1 level

So the emptiness = “category level does not exist”, NOT “unknown”
We need to fillna with 'NA' to signal that this is a valid category level that is explicitly ‘Not Applicable’ or missing.”

In [57]:
products['category_main'] = products['category_main'].fillna('NA')
products['category_mid'] = products['category_mid'].fillna('NA')
products['category_sub'] = products['category_sub'].fillna('NA')

This encoding helps with:
- Tree models (XGBoost, LightGBM) handle it cleanly
- One-hot encoding won’t crash
- Category hierarchy remains interpretable
- No accidental leakage of “Unknown” meaning (wrong semantic interpretation)

In [58]:
# Step 6: Round prices to 2 decimal places
products['price_original'] = products['price_original'].round(2)
products['price_discounted'] = products['price_discounted'].round(2)

# Step 7: Add a discount percentage column
# This tells us how much of a deal each product is 
# Formula: (original - discounted) / original * 100
# We use np.where to avoid division by zero (if original price is somehow 0)
products['discount_pct'] = np.where(
    products['price_original'] > 0,
    ((products['price_original'] - products['price_discounted']) / products['price_original'] * 100).round(1),
    0
)

print("Discount percentage distribution:")
print(products['discount_pct'].describe())
print(f"\nProducts with no discount (0%): {(products['discount_pct'] == 0).sum()}")
print(f"Products with >50% discount: {(products['discount_pct'] > 50).sum()}")

Discount percentage distribution:
count    14784.000000
mean        29.991761
std         17.065142
min          0.000000
25%         20.000000
50%         20.000000
75%         43.000000
max        100.000000
Name: discount_pct, dtype: float64

Products with no discount (0%): 802
Products with >50% discount: 2145


In [59]:
# Flag pricing anomalies
products['price_anomaly'] = products['discount_pct'] < 0

anomalies = products[products['price_anomaly']]
print(f'Products with discounted_price > original_price (data error): {len(anomalies)}')
if len(anomalies) > 0:
    print(anomalies[['product_code', 'product_name', 'price_original', 'price_discounted', 'discount_pct']].head(10))
    print()
    # For these, we swap the prices — most likely they were entered backwards
    # But discuss with the team first as there may be a business reason.
    print('ACTION: Swapping price_original and price_discounted for flagged rows.')
    print('If there is a business reason for these values, skip the swap.')
    swap_mask = products['price_anomaly']
    products.loc[swap_mask, ['price_original', 'price_discounted']] = (
        products.loc[swap_mask, ['price_discounted', 'price_original']].values
    )
    # Recalculate after swap
    products['discount_pct'] = np.where(
        products['price_original'] > 0,
        ((products['price_original'] - products['price_discounted']) / products['price_original'] * 100).round(1),
        0
    )
    print(f'After swap — negative discounts remaining: {(products["discount_pct"] < 0).sum()}')

print(f'\nDiscount % range: {products["discount_pct"].min()} to {products["discount_pct"].max()}')

Products with discounted_price > original_price (data error): 0

Discount % range: 0.0 to 100.0


In [60]:
# Step 7: Check for duplicate products
dupe_codes = products[products.duplicated('product_code', keep=False)]
print(f"Duplicate product codes: {len(dupe_codes)}")
if len(dupe_codes) > 0:
    print(dupe_codes[['product_code', 'product_name']])

Duplicate product codes: 0


In [61]:
# Step 8: Identify unpurchased products (cold-start problem)

all_product_codes = set(products['product_code'].unique())
purchased_codes = set(orders['product_code'].unique())

purchased = all_product_codes & purchased_codes  # intersection: number of ordered products that actually exist in the catalog
unpurchased = all_product_codes - purchased_codes  # difference

products['ever_purchased'] = products['product_code'].isin(all_product_codes & purchased_codes)

# Check orders that contain “invalid” product codes
invalid_in_orders = purchased_codes - all_product_codes
print("Unique product codes in orders but NOT in products table:", len(invalid_in_orders))

# optional: peek
print(list(sorted(invalid_in_orders))[:20])

Unique product codes in orders but NOT in products table: 6091
[29, 30, 41, 86, 97, 98, 99, 104, 107, 114, 115, 121, 124, 143, 161, 162, 181, 213, 239, 241]


We did some sanity checks and realized that 6091 unique product codes in orders that don’t exist in the product catalog and found that it was due to formatting issues and old catalog versions. 

In [69]:
# Fix/Normalize codes in both tables
products['product_code'] = products['product_code'].astype(str).str.strip().str.upper()
orders['product_code'] = orders['product_code'].astype(str).str.strip().str.upper()

In [70]:
# Keep only orders with valid catalog products
valid_product_set = set(products['product_code'])

orders = orders[orders['product_code'].isin(valid_product_set)].copy()

print("Clean orders:", len(orders))
print("Dropped invalid product rows:", len(orders) - len(orders))

Clean orders: 97702
Dropped invalid product rows: 0


In [71]:
# sanity checks for data integrity issues
print("Catalog size:", len(all_product_codes))
print("Purchased (catalog & orders):", len(all_product_codes & purchased_codes))
print("Unpurchased (catalog only):", len(all_product_codes - purchased_codes))
print("Invalid codes in orders (NOT in catalog):", len(purchased_codes - all_product_codes))

# Must always be true:
assert len(all_product_codes) == len(all_product_codes & purchased_codes) + len(all_product_codes - purchased_codes)

Catalog size: 14784
Purchased (catalog & orders): 10135
Unpurchased (catalog only): 4649
Invalid codes in orders (NOT in catalog): 0


In [72]:
# ~4,000 products have no purchase history — collaborative filtering can't help here.
all_product_codes = set(products['product_code'].unique())
purchased_codes = set(orders['product_code'].unique())

purchased = all_product_codes & purchased_codes  # intersection: number of ordered products that actually exist in the catalog
unpurchased = all_product_codes - purchased_codes  # difference

products['ever_purchased'] = products['product_code'].isin(all_product_codes & purchased_codes)

print(f'Total products in catalog: {len(all_product_codes)}')
print(f'Products with purchase history: {len(purchased)}')
print(f'Products NEVER purchased (cold-start): {len(unpurchased)}')
print(f'Cold-start % of catalog: {len(unpurchased)/len(all_product_codes)*100:.1f}%')
print()
print('NOTE: Cold-start products can only be recommended via content-based methods (category/brand similarity) until they accumulate purchase history.')

Total products in catalog: 14784
Products with purchase history: 10135
Products NEVER purchased (cold-start): 4649
Cold-start % of catalog: 31.4%

NOTE: Cold-start products can only be recommended via content-based methods (category/brand similarity) until they accumulate purchase history.


As we see here, even after normalizing the data, we are still seeing some missing/invalid codes in the orders table. We would consider this as a data integrity issue instead of a math issue or cleaning problem. This might be due to old/retired products still in transaction logs, missing rows in product catalog table, data entry or ETL mismatch. If we don't fix this problem, we will run into problems like creating item embeddings for non-existent products, breaking the user-item matrix alignment, or introducing noise in recommendation logic.

In [73]:
# Step 10: Summary of brands and categories
print('=== PRODUCT CATALOG SUMMARY ===')
print(f'Total products: {len(products)}')

print("=== BRAND BREAKDOWN ===")
print(products['brand'].value_counts())

print("\n=== MAIN CATEGORY BREAKDOWN ===")
print(products['category_main'].value_counts())

print("\n=== PRICE RANGE BY MAIN CATEGORY ===")
print(products.groupby('category_main')['price_discounted'].agg(['min', 'mean', 'max']).round(2))

=== PRODUCT CATALOG SUMMARY ===
Total products: 14784
=== BRAND BREAKDOWN ===
brand
Pastel                  815
Dior                    790
Flormar                 707
Golden Rose             647
YSL                     445
                       ... 
CRU Organics              1
Beauty Care               1
Syoss                     1
Joop                      1
Loreal Paris Recital      1
Name: count, Length: 270, dtype: int64

=== MAIN CATEGORY BREAKDOWN ===
category_main
Makyaj           5793
Parfüm           2658
Saç Bakım        1826
Kişisel Bakım    1703
Cilt Bakım       1131
Tekstil          1116
nan               229
Erkek Bakım       178
Bebek             150
Name: count, dtype: int64

=== PRICE RANGE BY MAIN CATEGORY ===
                 min     mean        max
category_main                           
Bebek           2.86   100.82     496.00
Cilt Bakım      0.00  4777.72  142436.25
Erkek Bakım    29.14   715.52    4966.67
Kişisel Bakım   0.00   358.43   15465.00
Makyaj        

In [74]:
# Final clean products table
print("CLEAN PRODUCTS TABLE:")
print(f"Shape: {products.shape}")
products

CLEAN PRODUCTS TABLE:
Shape: (14784, 12)


,product_name,product_code,category_main,category_mid,category_sub,full_path,price_original,price_discounted,brand,discount_pct,price_anomaly,ever_purchased
0,Sesu Sir Ağda Bantları Normal Tüyler 12'li,24568,Kişisel Bakım,Epilasyon Ürünleri,Ağda Bantları,Kişisel Bakım > Epilasyon Ürünleri > Ağda Bant...,0.00,0.00,Sesu,0.0,False,False
1,Sesu Vücut Ağda Bandı Kalın Tüy 12'li,24569,Kişisel Bakım,Epilasyon Ürünleri,Ağda Bantları,Kişisel Bakım > Epilasyon Ürünleri > Ağda Bant...,0.00,0.00,Sesu,0.0,False,False
2,YSL La Vestiaire Universite Unisex Parfüm Edp ...,117829,Parfüm,Unisex Parfüm,EDP Parfüm,Parfüm > Unisex Parfüm >EDP Parfüm,0.00,0.00,YSL,0.0,False,False
3,Pastel Show By Eyeliner 124,1458-907,Makyaj,Göz,Eyeliner,Makyaj > Göz > Eyeliner,0.00,0.00,Pastel,0.0,False,False
4,L'Oréal Paris Infaillible 24H Matte Cover Foun...,1135-1044,Makyaj,Yüz,Fondöten,Makyaj > Yüz > Fondöten,0.00,0.00,Loreal Paris Makyaj,0.0,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...
14779,La Prairie Platinum Rare Haute-Rejuvenation Mask,127565,Cilt Bakım,,Maskeler,Cilt Bakım > Maskeler,80625.00,80625.00,La Prairie,0.0,False,False
14780,La Prairie Platinum Rare Haute-Rejuvenation Cr...,127524,Cilt Bakım,Nemlendirici,Gündüz Kremi,Cilt Bakım > Nemlendirici > Gündüz Kremi,80833.33,80833.33,La Prairie,0.0,False,False
14781,La Prairie Life Matrix Haute Rejuveantion Cream,129430,Cilt Bakım,Nemlendirici,Gündüz Kremi,Cilt Bakım > Nemlendirici > Gündüz Kremi,87958.33,87958.33,La Prairie,0.0,False,True
14782,La Prairie Platinum Rare Haute-Rejuvenation Pr...,127531,Cilt Bakım,,Serumlar,Cilt Bakım > Serumlar,104250.00,104250.00,La Prairie,0.0,False,True


Note: Some categories have 1 some 2 some 3 category names meaning parfume>women parfume>edp parfume or just sets or personal care>mouth care which results in empty columns if there is 2 the middle category is empty if there is 1 only subcategory is filled 

In [75]:
# Save to CSV
customers.to_csv('clean_customers.csv', index=False)
orders.to_csv('clean_orders.csv', index=False)
products.to_csv('clean_products.csv', index=False)

## Joining Cleaned Tables
clean_orders
- customer_id
- product_code
- order_id (fact key)

clean_products
- product_code ← matches orders

clean_customers
- customer_id ← matches orders (members only, likely)

In [76]:
clean_orders = pd.read_csv("clean_orders.csv")
clean_products = pd.read_csv("clean_products.csv")
clean_customers = pd.read_csv("clean_customers.csv")

In [77]:
print("Orders shape:", clean_orders.shape)
print("Products shape:", clean_products.shape)
print("Customers shape:", clean_customers.shape)

print("\nOrders columns:", clean_orders.columns.tolist())
print("Products columns:", clean_products.columns.tolist())
print("Customers columns:", clean_customers.columns.tolist())

Orders shape: (97702, 11)
Products shape: (14784, 12)
Customers shape: (18070, 7)

Orders columns: ['customer_id', 'order_id', 'product_code', 'product_name', 'quantity', 'unit_price', 'purchase_date', 'line_total', 'is_gift', 'is_subproduct', 'is_member']
Products columns: ['product_name', 'product_code', 'category_main', 'category_mid', 'category_sub', 'full_path', 'price_original', 'price_discounted', 'brand', 'discount_pct', 'price_anomaly', 'ever_purchased']
Customers columns: ['customer_id', 'total_orders', 'total_products_bought', 'total_spend_tl', 'tenure_bucket', 'est_returns', 'has_returned']


In [78]:
clean_orders.head()

,customer_id,order_id,product_code,product_name,quantity,unit_price,purchase_date,line_total,is_gift,is_subproduct,is_member
0,T24540,TS010135579,362,Clinique For Men Kırışık Karşıtı Göz Kremi 15 Ml,1,756.0,2024-01-01,756.0,False,False,True
1,T24667,TS010135506,1212,Lancome Hypnose Kadın Parfüm Edp 75 Ml,1,2990.0,2024-01-01,2990.0,False,False,True
2,T24668,TS010135510,1970,Carolina Herrera 212 Sexy Men Erkek Parfüm Edt...,1,2449.6,2024-01-01,2449.6,False,False,True
3,ts_1704131707_8429,TS010135585,1970,Carolina Herrera 212 Sexy Men Erkek Parfüm Edt...,1,2449.6,2024-01-01,2449.6,False,False,False
4,T1186,TS010135548,3480,YSL La Nuit De L'Homme Erkek Deo Stick 75 Gr,1,916.5,2024-01-01,916.5,False,False,True


In [79]:
clean_products.head()

,product_name,product_code,category_main,category_mid,category_sub,full_path,price_original,price_discounted,brand,discount_pct,price_anomaly,ever_purchased
0,Sesu Sir Ağda Bantları Normal Tüyler 12'li,24568,Kişisel Bakım,Epilasyon Ürünleri,Ağda Bantları,Kişisel Bakım > Epilasyon Ürünleri > Ağda Bant...,0.0,0.0,Sesu,0.0,False,False
1,Sesu Vücut Ağda Bandı Kalın Tüy 12'li,24569,Kişisel Bakım,Epilasyon Ürünleri,Ağda Bantları,Kişisel Bakım > Epilasyon Ürünleri > Ağda Bant...,0.0,0.0,Sesu,0.0,False,False
2,YSL La Vestiaire Universite Unisex Parfüm Edp ...,117829,Parfüm,Unisex Parfüm,EDP Parfüm,Parfüm > Unisex Parfüm >EDP Parfüm,0.0,0.0,YSL,0.0,False,False
3,Pastel Show By Eyeliner 124,1458-907,Makyaj,Göz,Eyeliner,Makyaj > Göz > Eyeliner,0.0,0.0,Pastel,0.0,False,False
4,L'Oréal Paris Infaillible 24H Matte Cover Foun...,1135-1044,Makyaj,Yüz,Fondöten,Makyaj > Yüz > Fondöten,0.0,0.0,Loreal Paris Makyaj,0.0,False,True


In [80]:
clean_customers.head()

,customer_id,total_orders,total_products_bought,total_spend_tl,tenure_bucket,est_returns,has_returned
0,T10002,2,3,2731.40,>3yr,0,False
1,T10008,4,134,8537.99,>3yr,0,False
2,T1002,2,3,4234.00,>3yr,0,False
3,T10060,4,4,10434.45,>3yr,0,False
4,T10061,34,494,56770.06,>3yr,0,False


Logic for joining tables:
- orders & product: left join on product code
- joining with customers: left join on customer id to keep transactions and all customers (members and non-members)

In [81]:
# Step 1: Join orders with product info
orders_products = clean_orders.merge(
    clean_products,
    on='product_code',
    how='left'  # keep all transactions
)

# Step 2: Join the result with customer features
final_df = orders_products.merge(
    clean_customers,
    on='customer_id',
    how='left'  # VERY important: keeps guest users (ts_...) who do not exist in customers table
)

# effective key is (order_id, product_code)

In [82]:
# Sanity check:
print(final_df.shape)
print(final_df[['category_main', 'total_spend_tl']].isna().mean())

(97702, 28)
category_main     0.013736
total_spend_tl    0.250128
dtype: float64


In [83]:
final_df.head()

,customer_id,order_id,product_code,product_name_x,quantity,unit_price,purchase_date,line_total,is_gift,is_subproduct,...,brand,discount_pct,price_anomaly,ever_purchased,total_orders,total_products_bought,total_spend_tl,tenure_bucket,est_returns,has_returned
0,T24540,TS010135579,362,Clinique For Men Kırışık Karşıtı Göz Kremi 15 Ml,1,756.0,2024-01-01,756.0,False,False,...,Clinique,20.0,False,True,1.0,5.0,1984.41,2_to_3yr,0.0,False
1,T24667,TS010135506,1212,Lancome Hypnose Kadın Parfüm Edp 75 Ml,1,2990.0,2024-01-01,2990.0,False,False,...,Lancome,20.0,False,True,1.0,1.0,2990.00,2_to_3yr,0.0,False
2,T24668,TS010135510,1970,Carolina Herrera 212 Sexy Men Erkek Parfüm Edt...,1,2449.6,2024-01-01,2449.6,False,False,...,Carolina Herrera,20.0,False,True,3.0,3.0,9788.10,2_to_3yr,0.0,False
3,ts_1704131707_8429,TS010135585,1970,Carolina Herrera 212 Sexy Men Erkek Parfüm Edt...,1,2449.6,2024-01-01,2449.6,False,False,...,Carolina Herrera,20.0,False,True,NaN,NaN,NaN,NaN,NaN,NaN
4,T1186,TS010135548,3480,YSL La Nuit De L'Homme Erkek Deo Stick 75 Gr,1,916.5,2024-01-01,916.5,False,False,...,YSL,20.0,False,True,184.0,1433.0,1503396.85,>3yr,0.0,False


## Cleaning & Inspecting Final Joined Table

In [84]:
final_df.isna().sum()

customer_id                  0
order_id                     0
product_code                 0
product_name_x               0
quantity                     0
unit_price                   0
purchase_date                0
line_total                   0
is_gift                      0
is_subproduct                0
is_member                    0
product_name_y               0
category_main             1342
category_mid             12421
category_sub                 0
full_path                    0
price_original               0
price_discounted             0
brand                        1
discount_pct                 0
price_anomaly                0
ever_purchased               0
total_orders             24438
total_products_bought    24438
total_spend_tl           24438
tenure_bucket            24438
est_returns              24438
has_returned             24438
dtype: int64

In [85]:
final_df['customer_id'].str.startswith('ts_').sum()

np.int64(21169)

MOST missing customer features are from guest users as expected who don't have any info in the customers table, but ~3,000 rows are missing for a DIFFERENT reason, which could be:
- Some registered customers exist in orders but not in clean_customers
- clean_customers is a filtered subset (e.g., only active members)

In [86]:
missing_customer_rows = final_df[final_df['total_orders'].isna()]

print("Missing customer rows:", len(missing_customer_rows))
print("Of those, guest users:",
      missing_customer_rows['customer_id'].str.startswith('ts_').sum())

print("Non-guest unmatched customers:",
      (~missing_customer_rows['customer_id'].str.startswith('ts_')).sum())

Missing customer rows: 24438
Of those, guest users: 21169
Non-guest unmatched customers: 3269


This means that clean_customers table does NOT cover all customers in the orders table and this is normal for real world datasets because customer feature table may only include users with profiles and orders table includes all purchasers (including sparse users).

As of now, we should not drop these rows because we want to preserve full transaction behavior without introducing selection bias, however, when we get into Collaborative Filtering ot build the recommender system, we might need to drop these rows.

Check: How many product_codes in orders did NOT successfully match a row in the products table during the join?

In [87]:
# Unmatched products after join (indicates product_code mismatch or unclean orders)
missing_products = final_df[final_df['category_main'].isna()]['product_code'].nunique()
print("Unmatched product codes:", missing_products)

Unmatched product codes: 155


In [88]:
final_df[final_df['category_main'].isna()][['product_code']].head(10)

,product_code
590,134412
690,134412
1151,134318
1255,134412
1256,134412
1949,134318
3179,134318
8951,134319
11614,125694
11615,125696


In [89]:
# Check how many product codes in orders still don't match clean_products
unmatched = set(orders['product_code']) - set(clean_products['product_code'])
print("Unmatched product codes:", len(unmatched))

# Peek at a few suspicious ones
list(unmatched)[:20]

Unmatched product codes: 0


[]

In [90]:
missing_codes = set(final_df.loc[final_df['category_main'].isna(), 'product_code'])
len(missing_codes)

155

In [91]:
# flag missing codes
final_df['missing_product_info'] = final_df['category_main'].isna()

In [92]:
print("Total rows:", len(final_df))
print("Unique product_code:", final_df['product_code'].nunique())
print("Unique order_id:", final_df['order_id'].nunique())

Total rows: 97702
Unique product_code: 10135
Unique order_id: 51216


Might need 2 tables for the two problems:
- customer basket analysis: customer ID and their basket
- recommender: composite primary key with customer ID + specific order

##